# Q1

## Q1 code

In [101]:
from pyspark.sql.functions import size
from pyspark.ml.functions import vector_to_array

def vec_len(df):
    return df.select(size(vector_to_array(col("features"))).alias("n")).first()["n"]

print("len(train_df)     =", vec_len(train_df))
print("len(validate_df)  =", vec_len(validate_df))
print("len(test_df)      =", vec_len(test_df))


len(train_df)     = 119
len(validate_df)  = 119
len(test_df)      = 119


In [100]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.sql.functions import *
from pyspark.sql.types import *
import numpy as np
from streaming import MDSWriter

col_names = ["duration","protocol_type","service","flag","src_bytes",
"dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
"logged_in","num_compromised","root_shell","su_attempted","num_root",
"num_file_creations","num_shells","num_access_files","num_outbound_cmds",
"is_host_login","is_guest_login","count","srv_count","serror_rate",
"srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
"diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
"dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
"dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
"dst_host_rerror_rate","dst_host_srv_rerror_rate","class","difficulty"]

nominal_cols = ['protocol_type','service','flag']
binary_cols = ['land','logged_in','root_shell','su_attempted','is_host_login','is_guest_login']
continuous_cols = ['duration','src_bytes','dst_bytes','wrong_fragment','urgent','hot',
'num_failed_logins','num_compromised','num_root','num_file_creations','num_shells',
'num_access_files','num_outbound_cmds','count','srv_count','serror_rate','srv_serror_rate',
'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate',
'dst_host_count','dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate']

class FeatureTypeCaster(Transformer):
    def __init__(self): 
        super().__init__()
    def _transform(self, df):
        out = df
        for c in binary_cols + continuous_cols:
            out = out.withColumn(c, col(c).cast(DoubleType()))
        return out

class ColumnDropper(Transformer):
    def __init__(self, columns_to_drop=None):
        super().__init__()
        self.columns_to_drop = columns_to_drop or []
    def _transform(self, df):
        out = df
        for c in self.columns_to_drop:
            out = out.drop(c)
        return out

spark = (SparkSession.builder
         .master("local[*]")
         .appName("NSL-KDD-Spark-PyTorch")
         .getOrCreate())

train_raw = spark.read.csv("KDDTrain+.txt", header=False).toDF(*col_names)
test_raw  = spark.read.csv("KDDTest+.txt",  header=False).toDF(*col_names)

DOS = ['back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb']
probing = ['ipsweep','nmap','portsweep','satan','mscan','saint']
U2R = ['buffer_overflow','loadmodule','perl','rootkit','ps','sqlattack','xterm']

def add_attack(df):
    c = lower(col("class"))
    return (df.withColumn(
        "attack",
        when(c=='normal','normal')
        .when(c.isin(DOS), 'DOS')
        .when(c.isin(probing), 'probing')
        .when(c.isin(U2R), 'U2R')
        .otherwise('R2L')
    ))

train_raw2 = add_attack(train_raw)
test_raw2 = add_attack(test_raw)

label_indexer = StringIndexer(inputCol="attack", outputCol="label", handleInvalid="keep")
label_model = label_indexer.fit(train_raw2)        
train_idx = label_model.transform(train_raw2)
test_idx = label_model.transform(test_raw2)
labels_list = label_model.labels   

def get_preprocess_pipeline():
    stage_typecaster = FeatureTypeCaster()
    nominal_id_cols = [x + "_index"   for x in nominal_cols]
    nominal_oh_cols = [x + "_encoded" for x in nominal_cols]
    stage_nominal_indexer = StringIndexer(inputCols=nominal_cols, outputCols=nominal_id_cols, handleInvalid="keep")
    stage_nominal_onehot  = OneHotEncoder(inputCols=nominal_id_cols, outputCols=nominal_oh_cols, handleInvalid="keep", dropLast=True)
    feature_cols = continuous_cols + binary_cols + nominal_oh_cols
    to_remove = ["dst_host_serror_rate","srv_serror_rate","dst_host_srv_serror_rate",
                 "srv_rerror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"]
    feature_cols = [c for c in feature_cols if c not in to_remove]
    
    
    stage_vec    = VectorAssembler(inputCols=feature_cols, outputCol="vectorized_features", handleInvalid="keep")
    stage_scaler = StandardScaler(inputCol="vectorized_features", outputCol="features")
    stage_drop   = ColumnDropper(columns_to_drop=nominal_cols + [x + "_index" for x in nominal_cols] +
                                 [x + "_encoded" for x in nominal_cols] +
                                 binary_cols + continuous_cols + ["vectorized_features","difficulty"])
    return Pipeline(stages=[stage_typecaster, stage_nominal_indexer, stage_nominal_onehot,
                            stage_vec, stage_scaler, stage_drop])

feat_pipe = get_preprocess_pipeline()
feat_model = feat_pipe.fit(train_idx)        
train_df = feat_model.transform(train_idx)
test_full = feat_model.transform(test_idx)

validate_df, test_df = test_full.randomSplit([0.5, 0.5], seed=42)

from pyspark.ml.functions import vector_to_array
from streaming.base.converters.dataframe_to_mds import dataframe_to_mds

to_array_udf = udf(lambda v: [float(x) for x in v], ArrayType(FloatType()))
def write_df_to_mds(out_dir, df):
    df2 = (df.select(
        vector_to_array(col("features")).cast(ArrayType(FloatType())).alias("features"),
        col("label").cast(LongType()).alias("label")
    ))

    # Reference: GPT told me to use relative paths and forward slashes instead of absolute paths 
    # to avoid the drive letter being misinterpreted as a cloud provider preflix
    remote_out = "./" + str(out_dir).strip().lstrip(".\\/")
    remote_out = remote_out.replace("\\", "/")
    local_tmp= "./mds_tmp".replace("\\", "/")

    os.makedirs(local_tmp,  exist_ok=True)
    os.makedirs(remote_out, exist_ok=True)
    
    # TA told me to use MDSWriter instead
    # Reference: https://docs.mosaicml.com/projects/streaming/en/stable/api_reference/generated/streaming.MDSWriter.html
    with MDSWriter(
        columns={"features": "ndarray:float32", "label": "int64"},
        out=(local_tmp, remote_out),
        compression=None,                   
        size_limit=256*1024*1024            
    ) as writer:
        for row in df2.toLocalIterator():
            feat = np.asarray(row["features"], dtype=np.float32)   
            lab= np.int64(row["label"])                         
            writer.write({"features": feat, "label": lab})

write_df_to_mds("./NSL-KDD/train_mds", train_df)
write_df_to_mds("./NSL-KDD/validate_mds", validate_df)
write_df_to_mds("./NSL-KDD/test_mds", test_df)

import torch
from torch.utils.data import DataLoader
import streaming
from streaming import StreamingDataset

train_ds = StreamingDataset(local="./NSL-KDD/train_mds", batch_size=4, shuffle=True)
train_loader = DataLoader(train_ds, batch_size=4, num_workers=1)

for i, batch in enumerate(train_loader):
    y_int = batch["label"].tolist()        
    y_name = [labels_list[int(k)] for k in y_int]
    print(f"Batch {i} features shape:", batch["features"].shape)
    print("labels (int): ", y_int)
    print("labels (name):", y_name)
    if i == 2:
        break


Batch 0 features shape: torch.Size([4, 119])
labels (int):  [0, 0, 0, 1]
labels (name): ['normal', 'normal', 'normal', 'DOS']
Batch 1 features shape: torch.Size([4, 119])
labels (int):  [0, 0, 0, 1]
labels (name): ['normal', 'normal', 'normal', 'DOS']
Batch 2 features shape: torch.Size([4, 119])
labels (int):  [0, 1, 0, 0]
labels (name): ['normal', 'DOS', 'normal', 'normal']


# Q2

## Q2 code

In [11]:
from torch import nn

class myMultiLayerPerceptron(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_dim, 20),
            nn.ReLU(),
            nn.Linear(20, 20),
            nn.ReLU(),
            nn.Linear(20, 20),
            nn.ReLU(),
            nn.Linear(20, 20),
            nn.ReLU(),
            nn.Linear(20, output_dim)
        )

    def forward(self, x):
        y = self.sequential(x)
        return y

mymodel = myMultiLayerPerceptron(1, 1)  # input_dim=1, output_dim=1
print(mymodel)


myMultiLayerPerceptron(
  (sequential): Sequential(
    (0): Linear(in_features=1, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=20, bias=True)
    (5): ReLU()
    (6): Linear(in_features=20, out_features=20, bias=True)
    (7): ReLU()
    (8): Linear(in_features=20, out_features=1, bias=True)
  )
)


# Q3

## Q3 code

In [112]:
import os, time, math, json, copy, random
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col
from pyspark.sql.types import LongType
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader,random_split
import matplotlib.pyplot as plt
from builtins import max as pymax

# let the process reproducible
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(labels_list)  
print("Classes:", labels_list)

# Read batch once to make sure the input_dim
_probe_bs = 8
_probe_ds = StreamingDataset(local="./NSL-KDD/train_mds", batch_size=_probe_bs, shuffle=True)
_probe_loader = DataLoader(_probe_ds, batch_size=_probe_bs, num_workers=1)
xb0 = next(iter(_probe_loader))["features"]
input_dim = xb0.shape[1]
num_classes = len(labels_list)
print("Detected input_dim:", input_dim, " num_classes:", num_classes)

# Read the train/validate/test datasets using StreamingDataset,
# Ensure batch_size is passed to both StreamingDataset and DataLoader to maintain consistency.
def make_loaders(batch_size=256, num_workers=1):
    train_ds = StreamingDataset(local="./NSL-KDD/train_mds",    batch_size=batch_size, shuffle=True)
    validate_ds = StreamingDataset(local="./NSL-KDD/validate_mds", batch_size=batch_size, shuffle=False)
    test_ds = StreamingDataset(local="./NSL-KDD/test_mds",     batch_size=batch_size, shuffle=False)

    train_loader = DataLoader(train_ds,    batch_size=batch_size, num_workers=1)
    val_loader = DataLoader(validate_ds, batch_size=batch_size, num_workers=1)
    test_loader = DataLoader(test_ds,     batch_size=batch_size, num_workers=1)
    return train_loader, val_loader, test_loader


# Define a fully-connected neural network 
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_width=128, num_layers=3, dropout=0.3):
        super().__init__()
        layers = []
        last = input_dim
        for _ in range(num_layers):
            layers += [nn.Linear(last, hidden_width), 
                       nn.ReLU()]
            if dropout > 0: layers += [nn.Dropout(dropout)]
            last = hidden_width
        layers += [nn.Linear(last, output_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

# The Evaluator: return the loss and acc
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total, correct, total_loss = 0, 0, 0.0
    for batch in loader:
        x = torch.as_tensor(batch["features"], dtype=torch.float32, device=device)
        y = torch.as_tensor(batch["label"], dtype=torch.long, device=device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)
    return total_loss / pymax(total, 1), correct / pymax(total, 1)

    
def train_one_run(
    lr=1e-3, batch_size=256, epochs=20, weight_decay=0.0, optimizer_name="adam",
    hidden_width=128, num_layers=3, dropout=0.0, eval_interval=None, save_dir="./checkpoints_q3"
):
    os.makedirs(save_dir, exist_ok=True)
    train_loader, val_loader, test_loader = make_loaders(batch_size=batch_size)

    model = MLP(input_dim, num_classes, hidden_width, num_layers, dropout).to(device)
    loss_fn = nn.CrossEntropyLoss()
    if optimizer_name.lower() == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.95)

    history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
    best = {"epoch":-1, "val_acc":-1.0, "state_dict":None}

    print(f"\Start training: lr={lr}, batch_size={batch_size}, epochs={epochs}, "
          f"opt={optimizer_name}, hidden={hidden_width}, layers={num_layers}, wd={weight_decay}, dropout={dropout}")

    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for step, batch in enumerate(train_loader, start=1):
            x = torch.tensor(batch["features"], dtype=torch.float32, device=device)
            y = torch.tensor(batch["label"], dtype=torch.long, device=device)

            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()

            running_loss += loss.item() * x.size(0)
            running_correct += (logits.argmax(dim=1) == y).sum().item()
            running_total += x.size(0)

            if eval_interval and (step % eval_interval == 0):
                train_loss_step = running_loss / running_total
                train_acc_step  = running_correct / running_total
                val_loss_step, val_acc_step = evaluate(model, val_loader, loss_fn)
                print(f"Epoch {epoch:02d}  Step {step:04d} | "
                      f"train_loss {train_loss_step:.4f} acc {train_acc_step:.4f} | "
                      f"val_loss {val_loss_step:.4f} acc {val_acc_step:.4f}")

        # print every epoch
        train_loss = running_loss / pymax(running_total,1)
        train_acc  = running_correct / pymax(running_total,1)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # record the best 
        if val_acc > best["val_acc"]:
            best["epoch"] = epoch
            best["val_acc"] = val_acc
            best["state_dict"] = copy.deepcopy(model.state_dict())
            torch.save(best["state_dict"], os.path.join(save_dir, "best.pt"))

        print(f"[Epoch {epoch:02d}] "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
              f"train_acc={train_acc:.4f} val_acc={val_acc:.4f} | best_val_acc={best['val_acc']:.4f}")

        scheduler.step()

    # plot
    def _plot_curve(values, title, path):
        plt.figure()
        plt.plot(range(1, len(values)+1), values)
        plt.xlabel("Epoch"); plt.ylabel(title); plt.title(title)
        plt.tight_layout(); plt.savefig(path); plt.close()

    _plot_curve(history["train_loss"], "Train Loss", os.path.join(save_dir, "train_loss.png"))
    _plot_curve(history["val_loss"],   "Val Loss",   os.path.join(save_dir, "val_loss.png"))
    _plot_curve(history["train_acc"],  "Train Accuracy", os.path.join(save_dir, "train_acc.png"))
    _plot_curve(history["val_acc"],    "Val Accuracy",   os.path.join(save_dir, "val_acc.png"))

    if best["state_dict"] is not None:
        model.load_state_dict(best["state_dict"])
    test_loss, test_acc = evaluate(model, test_loader, loss_fn)

    print(f"Best epoch={best['epoch']}, best val acc={best['val_acc']:.4f}")
    print(f"Test  loss={test_loss:.4f}, Test acc={test_acc:.4f}")
    meta = dict(
        lr=lr, batch_size=batch_size, epochs=epochs, optimizer=optimizer_name,
        hidden_width=hidden_width, num_layers=num_layers, weight_decay=weight_decay,
        dropout=dropout, best_epoch=best["epoch"], best_val_acc=best["val_acc"],
        test_loss=test_loss, test_acc=test_acc
    )
    with open(os.path.join(save_dir, "run_summary.json"), "w") as f:
        json.dump(meta, f, indent=2)
    return meta

Classes: ['normal', 'DOS', 'probing', 'R2L', 'U2R']
Detected input_dim: 113  num_classes: 5


In [113]:
q3_meta = train_one_run(
    lr=3e-4, batch_size=512, epochs=5,
    optimizer_name="adam", hidden_width=128, num_layers=4,
    weight_decay=1e-4, dropout=0.4,
    eval_interval=300, save_dir="./checkpoints_q3"
)
q3_meta

\Start training: lr=0.0003, batch_size=512, epochs=5, opt=adam, hidden=128, layers=4, wd=0.0001, dropout=0.4


C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\799508932.py:103: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(batch["features"], dtype=torch.float32, device=device)
C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\799508932.py:104: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(batch["label"], dtype=torch.long, device=device)


[Epoch 01] train_loss=0.4866 val_loss=1.5650 | train_acc=0.8420 val_acc=0.7114 | best_val_acc=0.7114
[Epoch 02] train_loss=0.1239 val_loss=1.5248 | train_acc=0.9679 val_acc=0.7110 | best_val_acc=0.7114
[Epoch 03] train_loss=0.0889 val_loss=1.6132 | train_acc=0.9738 val_acc=0.7360 | best_val_acc=0.7360
[Epoch 04] train_loss=0.0678 val_loss=1.4780 | train_acc=0.9785 val_acc=0.7498 | best_val_acc=0.7498
[Epoch 05] train_loss=0.0566 val_loss=1.6226 | train_acc=0.9816 val_acc=0.7675 | best_val_acc=0.7675
Best epoch=5, best val acc=0.7675
Test  loss=1.5832, Test acc=0.7647


{'lr': 0.0003,
 'batch_size': 512,
 'epochs': 5,
 'optimizer': 'adam',
 'hidden_width': 128,
 'num_layers': 4,
 'weight_decay': 0.0001,
 'dropout': 0.4,
 'best_epoch': 5,
 'best_val_acc': 0.7674623447546904,
 'test_loss': 1.5831561222420802,
 'test_acc': 0.7647216513269591}

## Q3 output

Refer to q3_1.png, q3_2.png, q3_3.png, q3_4.png

# Q4

## Q4-Trial 1

In [55]:
trial1 = train_one_run(
    lr=3e-4, batch_size=512, epochs=10,
    optimizer_name="adam", hidden_width=256, num_layers=3,
    weight_decay=1e-4, dropout=0.0,
    eval_interval=300, save_dir="./Q4_trial1_baseline"
)
trial1


[Q3] Start training: lr=0.0003, batch_size=512, epochs=10, opt=adam, hidden=256, layers=3, wd=0.0001, dropout=0.0


C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\2433357736.py:103: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(batch["features"], dtype=torch.float32, device=device)
C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\2433357736.py:104: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(batch["label"], dtype=torch.long, device=device)


[Epoch 01] train_loss=0.2514 val_loss=1.4344 | train_acc=0.9422 val_acc=0.7391 | best_val_acc=0.7391
[Epoch 02] train_loss=0.0408 val_loss=1.7494 | train_acc=0.9872 val_acc=0.7578 | best_val_acc=0.7578
[Epoch 03] train_loss=0.0299 val_loss=2.0128 | train_acc=0.9908 val_acc=0.7799 | best_val_acc=0.7799
[Epoch 04] train_loss=0.0250 val_loss=1.9725 | train_acc=0.9923 val_acc=0.7823 | best_val_acc=0.7823
[Epoch 05] train_loss=0.0232 val_loss=2.0788 | train_acc=0.9926 val_acc=0.7754 | best_val_acc=0.7823
[Epoch 06] train_loss=0.0217 val_loss=2.1048 | train_acc=0.9932 val_acc=0.7690 | best_val_acc=0.7823
[Epoch 07] train_loss=0.0208 val_loss=2.2300 | train_acc=0.9937 val_acc=0.7701 | best_val_acc=0.7823
[Epoch 08] train_loss=0.0197 val_loss=2.3375 | train_acc=0.9941 val_acc=0.7700 | best_val_acc=0.7823
[Epoch 09] train_loss=0.0192 val_loss=2.1973 | train_acc=0.9941 val_acc=0.7693 | best_val_acc=0.7823
[Epoch 10] train_loss=0.0182 val_loss=2.2796 | train_acc=0.9943 val_acc=0.7724 | best_val_a

{'lr': 0.0003,
 'batch_size': 512,
 'epochs': 10,
 'optimizer': 'adam',
 'hidden_width': 256,
 'num_layers': 3,
 'weight_decay': 0.0001,
 'dropout': 0.0,
 'best_epoch': 4,
 'best_val_acc': 0.782348277988197,
 'test_loss': 1.9733683073480581,
 'test_acc': 0.7825931552140113}

## Trial 2

In [59]:
trial2 = train_one_run(
    lr=3e-4, batch_size=512, epochs=10,
    optimizer_name="adam", hidden_width=256, num_layers=3,
    weight_decay=5e-4, dropout=0.3,
    eval_interval=300, save_dir="./Q4_trial2_reg"
)


\Start training: lr=0.0003, batch_size=512, epochs=10, opt=adam, hidden=256, layers=3, wd=0.0005, dropout=0.3


C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\4172859292.py:103: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(batch["features"], dtype=torch.float32, device=device)
C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\4172859292.py:104: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(batch["label"], dtype=torch.long, device=device)


[Epoch 01] train_loss=0.2743 val_loss=1.5279 | train_acc=0.9260 val_acc=0.7402 | best_val_acc=0.7402
[Epoch 02] train_loss=0.0677 val_loss=1.5333 | train_acc=0.9776 val_acc=0.7570 | best_val_acc=0.7570
[Epoch 03] train_loss=0.0463 val_loss=1.7519 | train_acc=0.9849 val_acc=0.7676 | best_val_acc=0.7676
[Epoch 04] train_loss=0.0394 val_loss=1.6201 | train_acc=0.9871 val_acc=0.7726 | best_val_acc=0.7726
[Epoch 05] train_loss=0.0357 val_loss=1.6753 | train_acc=0.9883 val_acc=0.7678 | best_val_acc=0.7726
[Epoch 06] train_loss=0.0327 val_loss=1.7714 | train_acc=0.9889 val_acc=0.7692 | best_val_acc=0.7726
[Epoch 07] train_loss=0.0316 val_loss=1.6687 | train_acc=0.9899 val_acc=0.7689 | best_val_acc=0.7726
[Epoch 08] train_loss=0.0297 val_loss=1.7656 | train_acc=0.9902 val_acc=0.7751 | best_val_acc=0.7751
[Epoch 09] train_loss=0.0288 val_loss=1.6157 | train_acc=0.9905 val_acc=0.7711 | best_val_acc=0.7751
[Epoch 10] train_loss=0.0273 val_loss=1.7142 | train_acc=0.9910 val_acc=0.7660 | best_val_a

## Trial 3

In [63]:
trial3 = train_one_run(
    lr=0.05, batch_size=64, epochs=20,
    optimizer_name="sgd",
    hidden_width=64, num_layers=3,
    weight_decay=1e-3, dropout=0.3,
    eval_interval=800, save_dir="./Q4_trial3_sgd"
)



\Start training: lr=0.05, batch_size=64, epochs=20, opt=sgd, hidden=64, layers=3, wd=0.001, dropout=0.3


C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\4172859292.py:103: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(batch["features"], dtype=torch.float32, device=device)
C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\4172859292.py:104: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(batch["label"], dtype=torch.long, device=device)


Epoch 01  Step 0800 | train_loss 0.1569 acc 0.9555 | val_loss 1.3175 acc 0.7395
Epoch 01  Step 1600 | train_loss 0.1020 acc 0.9693 | val_loss 1.4291 acc 0.7579
[Epoch 01] train_loss=0.0915 val_loss=1.5974 | train_acc=0.9724 val_acc=0.7564 | best_val_acc=0.7564
Epoch 02  Step 0800 | train_loss 0.0702 acc 0.9775 | val_loss 1.1357 acc 0.7432
Epoch 02  Step 1600 | train_loss 0.0550 acc 0.9820 | val_loss 1.2615 acc 0.7702
[Epoch 02] train_loss=0.0521 val_loss=1.0293 | train_acc=0.9826 val_acc=0.7762 | best_val_acc=0.7762
Epoch 03  Step 0800 | train_loss 0.0672 acc 0.9792 | val_loss 1.0040 acc 0.7380
Epoch 03  Step 1600 | train_loss 0.0514 acc 0.9834 | val_loss 1.2817 acc 0.7640
[Epoch 03] train_loss=0.0486 val_loss=1.1129 | train_acc=0.9843 val_acc=0.7843 | best_val_acc=0.7843
Epoch 04  Step 0800 | train_loss 0.0611 acc 0.9811 | val_loss 1.4004 acc 0.7710
Epoch 04  Step 1600 | train_loss 0.0499 acc 0.9838 | val_loss 1.3025 acc 0.7750
[Epoch 04] train_loss=0.0476 val_loss=1.2045 | train_acc=

## Discussion

In Trial 1, I established a new baseline by training with the Adam optimizer using a learning rate of 3e-4, batch size 512, and no regularization.
This configuration converged quickly but showed noticeable overfitting — training accuracy reached ≈ 0.99 while validation accuracy plateaued around 0.78 and validation loss kept rising.

In Trial 2, I introduced dropout (0.3) and L2 weight decay (5e-4) to regularize the model while keeping the same optimizer and architecture.
This reduced the training-validation gap and produced smoother learning curves, but validation accuracy remained near 0.775, implying limited effect from explicit regularization alone.

In Trial 3, I switched from Adam to SGD with a higher learning rate (0.05), smaller batch size (64), and stronger weight decay (1e-3).
This combination added implicit regularization via gradient noise and led to a more stable convergence.
The model achieved a best validation accuracy of 0.799 and test accuracy of 0.7999, with validation loss flattening around epoch 8 — indicating better generalization than the previous Adam-based settings.

## Q4 output

-   Trial1 refers to q4_1_(1-4).png
-   Trial2 refers to q4_2_(1-4).png
-   Trial3 refers to q4_3_(1-4).png

# Q5

In [75]:
import os, json, torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the best model (from Q4_trial3_sgd) and evaluate on test set
run_dir = r"C:\Users\yiyin\Desktop\14763A\hwk\hwk6\Q4_trial3_sgd"  
checkpoint_path = os.path.join(run_dir, "best.pt")
meta_path = os.path.join(run_dir, "run_summary.json")

# Read the parameters
with open(meta_path, "r") as f:
    meta = json.load(f)

hidden_width = meta.get("hidden_width", 128)
num_layers = meta.get("num_layers", 3)
dropout = meta.get("dropout", 0.0)
batch_size = meta.get("batch_size", 256)

model = MLP(input_dim, num_classes,
            hidden_width=hidden_width,
            num_layers=num_layers,
            dropout=dropout).to(device)

state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict, strict=True)

# Reload the test dataset using the same batch size and define the loss function (CrossEntropyLoss) for evaluation.
_, _, test_loader = make_loaders(batch_size=batch_size)
loss_fn = nn.CrossEntropyLoss()

# switch to evaluation mode
model.eval()                              
# disable gradient tracking
# Reference:https://www.zywvvd.com/notes/study/deep-learning/pytorch/train-eval-nograd/train-eval-nograd/
with torch.no_grad():                     
    test_loss, test_acc = evaluate(model, test_loader, loss_fn)

print(f"Test Loss = {test_loss:.4f}")
print(f"Test Acc = {test_acc:.4f}")


C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\1248642550.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location=devic

Test Loss = 1.1528
Test Acc = 0.7999


# Discussion

During the evaluation process, when we want to prevent PyTorch from building the computation graph, we can switch the model to eval mode using model.eval()(which is from the lecture 15).
Also, we could disable autograd by using [with torch.no_grad():].

## Q5 output

Refer to q5.png

# Q6

# Q7

## Q7 code

In [90]:
import torch
from torch import nn

class myBaseModel(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.sequential = nn.Sequential(  # here we stack multiple layers together
            nn.Linear(input_dim,20),
            nn.Tanh(), # Using Tanh activation!
            nn.Linear(20,20),
            nn.Tanh(),
            nn.Linear(20,20),
            nn.Tanh(),
            nn.Linear(20,20),
            nn.Tanh(),
            nn.Linear(20,output_dim)
        )
    def forward(self,x):
        y = self.sequential(x)
        return y

def get_base_model():
    base_model = myBaseModel(113,2)
    base_model.load_state_dict(torch.load("hw6q7_basemodel"))
    return base_model

In [110]:
from pyspark.ml.feature import VectorAssembler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 5 

def evaluate(model, data_loader, device):
    model.eval()
    crit = nn.CrossEntropyLoss()
    tot, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for batch in data_loader:
            x = batch["features"].to(device)
            y = batch["label"].long().to(device)
            x = x.index_select(dim=1, index=keep_idx)
            logits = model(x)
            loss = crit(logits, y)
            loss_sum += loss.item() * y.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            tot += y.size(0)
    return loss_sum / tot, correct / tot

q7model = get_base_model().to(device)

assert isinstance(q7model.sequential[-1], nn.Linear)
in_features = q7model.sequential[-1].in_features
q7model.sequential[-1] = nn.Linear(in_features, num_classes).to(device)

print(q7model)

myBaseModel(
  (sequential): Sequential(
    (0): Linear(in_features=113, out_features=20, bias=True)
    (1): Tanh()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): Tanh()
    (4): Linear(in_features=20, out_features=20, bias=True)
    (5): Tanh()
    (6): Linear(in_features=20, out_features=20, bias=True)
    (7): Tanh()
    (8): Linear(in_features=20, out_features=5, bias=True)
  )
)


C:\Users\yiyin\AppData\Local\Temp\ipykernel_25512\3918244055.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  base_model.load_state_dict(torch.load("hw6q7_basemodel"))


In [111]:
with torch.no_grad():
    for batch in test_loader:   
        x = batch["features"].to(device)
        y = batch["label"].long().to(device)
        logits = q7model(x)
        loss = criterion(logits, y)
        test_loss += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

test_loss /= total
test_acc = correct / total
print(f"=== Q7-1 Test metric BEFORE fine-tuning ===")
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x119 and 113x20)